# Four-Way Comparative Analysis: Quantum Transport Impurity Concentration Prediction
This notebook benchmarks four distinct approaches for predicting impurity concentration in **7-AGNR (Armchair Graphene Nanoribbons)** from their normalized transmission spectra $T(E)$:

1. **Physics-Informed Neural Network (ConductanceMLP + Curvature Misfit)**: Fully connected network with LayerNorm regularized by an adaptive curvature-weighted misfit loss.
2. **1D-Patched Vision Transformer (PatchedTransformerV2)**: Global self-attention model capturing non-local spectral correlations and resonance interference across distant energy bands.
3. **Gradient Boosted Trees (XGBoost Regressor)**: Hardware-accelerated histogram decision trees trained on normalized conductance spectra.
4. **Physical Misfit Baseline**: Analytical baseline predicting concentration by finding the minimum squared difference against configuration-averaged reference spectra.


In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import xgboost as xgb
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


In [ ]:
# --- 1. ConductanceMLP (PINN Backbone) ---
class ConductanceMLP(nn.Module):
    """Multi-Layer Perceptron for concentration prediction (from pinn_agnr_curvature.py)."""
    def __init__(self, input_length: int = 200, hidden_dims: list = None,
                 dropout: float = 0.2, noise_std: float = 0.02):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [256, 128, 64, 32]
        self.noise_std = noise_std
        layers = []
        in_dim = input_length
        for h_dim in hidden_dims:
            layers.append(nn.Linear(in_dim, h_dim))
            layers.append(nn.LayerNorm(h_dim))
            layers.append(nn.ReLU(inplace=True))
            layers.append(nn.Dropout(dropout))
            in_dim = h_dim
        self.mlp = nn.Sequential(*layers)
        self.regressor = nn.Linear(in_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.training and self.noise_std > 0:
            noise = 1.0 + torch.randn_like(x) * self.noise_std
            x = x * noise
        x = self.mlp(x)
        return self.regressor(x)


# --- 2. Patched Transformer v2 Components ---
class ConvStem(nn.Module):
    def __init__(self, in_channels=1, mid_channels=16, out_channels=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, mid_channels, kernel_size=7, stride=1, padding=3),
            nn.GELU(),
            nn.Conv1d(mid_channels, out_channels, kernel_size=5, stride=1, padding=2),
            nn.GELU(),
        )

    def forward(self, x):
        return self.net(x)


class PatchEmbedding1D(nn.Module):
    def __init__(self, seq_len=200, patch_size=10, in_channels=32, embed_dim=128):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = seq_len // patch_size
        self.proj = nn.Conv1d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = nn.LayerNorm(embed_dim)
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches, embed_dim) * 0.02)

    def forward(self, x):
        x = self.proj(x).transpose(1, 2)
        return self.norm(x) + self.pos_embed


class DropPath(nn.Module):
    def __init__(self, drop_prob=0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if not self.training or self.drop_prob == 0.0:
            return x
        keep = 1.0 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        mask = torch.bernoulli(torch.full(shape, keep, device=x.device))
        return x * mask / keep


class TransformerBlock(nn.Module):
    def __init__(self, embed_dim=128, num_heads=4, mlp_ratio=4.0, dropout=0.1, drop_path=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.drop_path1 = DropPath(drop_path)
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, embed_dim),
            nn.Dropout(dropout),
        )
        self.drop_path2 = DropPath(drop_path)

    def forward(self, x):
        x_norm = self.norm1(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        x = x + self.drop_path1(attn_out)
        return x + self.drop_path2(self.mlp(self.norm2(x)))


class PatchedTransformerV2(nn.Module):
    def __init__(self, seq_len=200, patch_size=10, stem_channels=32, embed_dim=128,
                 depth=4, num_heads=4, mlp_ratio=4.0, dropout=0.1,
                 drop_path_rate=0.1, noise_std=0.02):
        super().__init__()
        self.noise_std = noise_std
        self.stem = ConvStem(1, 16, stem_channels)
        self.patch_embed = PatchEmbedding1D(seq_len, patch_size, stem_channels, embed_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, depth)]
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, dropout, dpr[i])
            for i in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim // 2, 1),
        )

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        if self.training and self.noise_std > 0:
            x = x * (1.0 + torch.randn_like(x) * self.noise_std)
        x = self.stem(x)
        x = self.patch_embed(x)
        B = x.shape[0]
        x = torch.cat([self.cls_token.expand(B, -1, -1), x], dim=1)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        return self.head(x[:, 0])


In [ ]:
# --- Load Pretrained Checkpoints ---
# 1. Load ConductanceMLP
mlp_save_path = 'pinn_agnr_curvature.pt'
mlp_checkpoint = torch.load(mlp_save_path, map_location=device, weights_only=False)
spectrum_len = mlp_checkpoint.get('args', {}).get('spectrum_len', 200)

mlp_model = ConductanceMLP(input_length=spectrum_len).to(device)
mlp_model.load_state_dict(mlp_checkpoint['model_state_dict'])
mlp_model.eval()
print("✓ ConductanceMLP (PINN) loaded successfully!")

# 2. Load PatchedTransformerV2
tf_save_path = 'patched_transformer_v2.pt'
tf_checkpoint = torch.load(tf_save_path, map_location=device, weights_only=False)
tf_args = tf_checkpoint.get('args', {})

tf_model = PatchedTransformerV2(
    seq_len=tf_args.get('spectrum_len', 200),
    patch_size=tf_args.get('patch_size', 10),
    stem_channels=32,
    embed_dim=tf_args.get('embed_dim', 128),
    depth=tf_args.get('depth', 4),
    num_heads=tf_args.get('num_heads', 4),
    mlp_ratio=4.0,
).to(device)

raw_sd = tf_checkpoint['model_state_dict']
cleaned_sd = {k.replace('_orig_mod.', ''): v for k, v in raw_sd.items()}
tf_model.load_state_dict(cleaned_sd)
tf_model.eval()
print("✓ PatchedTransformerV2 loaded successfully!")


In [ ]:
project_root = Path("../../")
data_dir = project_root / "data" / "raw" / "transmission_results"
test_dir = project_root / "data" / "test" / "transmission_results"

# Load pristine
pristine_path = data_dir / "pristine.npy"
pristine = np.load(str(pristine_path))[:spectrum_len].astype(np.float32)

# Build reference spectra for misfit & training set for XGBoost
conc_range = np.arange(3, 45, 2)
n_sample_configs = 100
ref_list = []
X_train, y_train = [], []

print("Building reference spectra and training data for XGBoost...")
for con in conc_range:
    acc = []
    for cfg in range(n_sample_configs):
        fpath = data_dir / f"7_agnr_conc{int(con)}_cfg{cfg}.npy"
        if fpath.exists():
            spec = np.load(str(fpath)).astype(np.float32)[:spectrum_len]
            spec = np.clip(spec, 0, pristine)
            acc.append(spec)
            X_train.append(spec / (pristine + 1e-8))
            y_train.append(con)
    if len(acc) > 0:
        avg_spec = np.mean(acc, axis=0)
        ref_list.append(avg_spec / (pristine + 1e-8))

ref_spectra = np.array(ref_list)
print(f"✓ Loaded {len(ref_list)} reference spectra for concentrations: {conc_range}")

# Train XGBoost Regressor
print("Fitting XGBoost Regressor...")
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(np.array(X_train), np.array(y_train))
print("✓ XGBoost Regressor fitted successfully!")


In [ ]:
def predict_misfit(test_spectrum_norm, ref_spectra, concentrations, spectrum_start=20, spectrum_end=150):
    """Predict concentration via argmin of squared error against reference library."""
    test_crop = test_spectrum_norm[spectrum_start:spectrum_end]
    ref_crop = ref_spectra[:, spectrum_start:spectrum_end]
    pristine_crop = pristine[spectrum_start:spectrum_end]
    
    # Un-normalize & clip
    test_unnorm = np.clip(test_crop * pristine_crop, 0.0, pristine_crop)
    ref_unnorm = ref_crop * pristine_crop
    
    # Squared difference
    diff = test_unnorm[None, :] - ref_unnorm
    mis = np.sum(diff ** 2, axis=1) / 150.0
    
    return concentrations[np.argmin(mis)], mis


In [ ]:
test_files = sorted(glob.glob(str(test_dir / "*.npy")))
print(f"Evaluating all 4 models on {len(test_files)} held-out test spectra...")

true_concs = []
mlp_preds = []
tf_preds = []
xgb_preds = []
misfit_preds = []

for fpath in test_files:
    filename = os.path.basename(fpath)
    true_c = float(filename.split('_conc')[1].split('_')[0])
    true_concs.append(true_c)
    
    # Load and preprocess
    spec = np.load(fpath).astype(np.float32)[:spectrum_len]
    spec = np.clip(spec, 0, pristine)
    spec_norm = spec / (pristine + 1e-8)
    
    # 1 & 2. PyTorch models (MLP and Transformer)
    with torch.no_grad():
        x = torch.tensor(spec_norm, dtype=torch.float32).unsqueeze(0).to(device)
        mlp_preds.append(mlp_model(x).item())
        tf_preds.append(tf_model(x).item())
        
    # 3. XGBoost
    xgb_preds.append(xgb_model.predict(spec_norm.reshape(1, -1))[0])
    
    # 4. Physical Misfit
    m_pred, _ = predict_misfit(spec_norm, ref_spectra, conc_range)
    misfit_preds.append(m_pred)

true_concs = np.array(true_concs)
mlp_preds = np.array(mlp_preds)
tf_preds = np.array(tf_preds)
xgb_preds = np.array(xgb_preds)
misfit_preds = np.array(misfit_preds)

# Compute metrics
def calc_metrics(pred, true):
    mae = np.mean(np.abs(pred - true))
    rmse = np.sqrt(np.mean((pred - true) ** 2))
    max_err = np.max(np.abs(pred - true))
    return mae, rmse, max_err

results_df = pd.DataFrame([
    {"Method": "Deep Learning (MLP)", **dict(zip(["MAE", "RMSE", "Max Error"], calc_metrics(mlp_preds, true_concs)))},
    {"Method": "Patched Transformer v2", **dict(zip(["MAE", "RMSE", "Max Error"], calc_metrics(tf_preds, true_concs)))},
    {"Method": "XGBoost Regressor", **dict(zip(["MAE", "RMSE", "Max Error"], calc_metrics(xgb_preds, true_concs)))},
    {"Method": "Physical Misfit Baseline", **dict(zip(["MAE", "RMSE", "Max Error"], calc_metrics(misfit_preds, true_concs)))},
])

print("=" * 65)
print(results_df.to_string(index=False))
print("=" * 65)


In [ ]:
# --- Scatter Plots: Predicted vs True ---
fig, axes = plt.subplots(1, 4, figsize=(24, 6))
models_data = [
    ("Deep Learning (MLP)", mlp_preds, "#2196F3"),
    ("Patched Transformer v2", tf_preds, "#FF9800"),
    ("XGBoost Regressor", xgb_preds, "#9C27B0"),
    ("Physical Misfit", misfit_preds, "#4CAF50"),
]
lo, hi = true_concs.min(), true_concs.max()

for ax, (name, preds, color) in zip(axes, models_data):
    mae, rmse, _ = calc_metrics(preds, true_concs)
    ax.scatter(true_concs, preds, alpha=0.4, s=18, color=color, edgecolors='none')
    ax.plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='Ideal 1:1')
    ax.set_title(f"{name}\nMAE: {mae:.3f} | RMSE: {rmse:.3f}", fontsize=12)
    ax.set_xlabel("True Concentration")
    ax.set_ylabel("Predicted Concentration")
    ax.set_xlim(lo - 2, hi + 2)
    ax.set_ylim(lo - 5, hi + 5)
    ax.grid(True, alpha=0.2)
    ax.legend(loc='upper left')

plt.suptitle("Predicted vs True Concentration Across 2,100 Test Spectra", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# --- Error Distribution Histograms ---
fig, ax = plt.subplots(figsize=(11, 5))
bins = np.linspace(-15, 15, 50)

for name, preds, color in models_data:
    mae, _, _ = calc_metrics(preds, true_concs)
    ax.hist(preds - true_concs, bins=bins, alpha=0.45, label=f'{name} (MAE={mae:.2f})', color=color, density=True)

ax.axvline(0, color='r', linestyle='--', lw=1.5)
ax.set_xlabel("Prediction Error (Predicted − True)")
ax.set_ylabel("Density")
ax.set_title("Error Distribution Comparison Across All 4 Models")
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()


In [ ]:
# --- Per-Concentration MAE Breakdown ---
unique_concs = np.sort(np.unique(true_concs))
per_conc_dict = {name: [] for name, _, _ in models_data}

for c in unique_concs:
    mask = true_concs == c
    for name, preds, _ in models_data:
        per_conc_dict[name].append(np.mean(np.abs(preds[mask] - c)))

fig, ax = plt.subplots(figsize=(14, 5))
w = 0.8
x = np.arange(len(unique_concs))

for idx, (name, _, color) in enumerate(models_data):
    offset = (idx - 1.5) * (w / 4)
    ax.bar(x + offset, per_conc_dict[name], width=w/4, label=name, color=color, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([f"{int(c)}" for c in unique_concs], fontsize=8)
ax.set_xlabel("True Concentration")
ax.set_ylabel("MAE (Impurity Count)")
ax.set_title("Per-Concentration MAE Breakdown Across All Models")
ax.legend()
ax.grid(True, alpha=0.2, axis='y')
plt.tight_layout()
plt.show()


In [ ]:
# --- Training Loss Convergence (MLP vs Transformer) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

mlp_train = mlp_checkpoint.get('train_losses', [])
mlp_val = mlp_checkpoint.get('val_losses', [])
if mlp_train:
    axes[0].plot(mlp_train, label='Train', color='#2196F3')
    axes[0].plot(mlp_val, label='Val', color='#2196F3', linestyle='--')
    axes[0].set_title("ConductanceMLP Training Curves")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.2)

tf_train = tf_checkpoint.get('train_losses', [])
tf_val = tf_checkpoint.get('val_losses', [])
if tf_train:
    axes[1].plot(tf_train, label='Train', color='#FF9800')
    axes[1].plot(tf_val, label='Val', color='#FF9800', linestyle='--')
    axes[1].set_title("PatchedTransformerV2 Training Curves")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    axes[1].grid(True, alpha=0.2)

plt.suptitle("Deep Learning Training Convergence", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
